# Логирование шагов и `raise` для контракта данных

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## 1. Лог шагов обработки

In [ ]:
log_steps = []
# добавьте минимум 5 шагов (строки)
assert len(log_steps) >= 5
print(log_steps)

## 2. Функция валидации

In [ ]:
def validate_orders(frame):
    # ваш код
    return True


ok = validate_orders(orders.merge(payments, on='order_id', how='left'))
assert ok is True
print(ok)

## 3. Проверка: отрицательная сумма -> raise

In [ ]:
bad_row = orders.merge(payments, on='order_id', how='left').head(3).copy()
bad_row.loc[bad_row.index[0], 'payment_value'] = -10
error_text = ''
try:
    validate_orders(bad_row)
except ValueError as e:
    error_text = str(e)
assert len(error_text) > 0
print(error_text)

## 4. Проверка: пропуск даты -> raise

In [ ]:
bad_dates = orders.merge(payments, on='order_id', how='left').head(5).copy()
bad_dates.loc[bad_dates.index[0], 'order_purchase_timestamp'] = pd.NaT
error_date = ''
try:
    validate_orders(bad_dates)
except ValueError as e:
    error_date = str(e)
assert len(error_date) > 0
print(error_date)

## 5. `LOG_NOTE`

In [ ]:
LOG_NOTE = ''
assert len(LOG_NOTE) > 120
print(LOG_NOTE)